In [1]:
!pip install torch torchaudio librosa audiomentations pandas scikit-learn tqdm

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/igraph-0.11.8-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_thunder-0.2.0.dev0-py3.12.egg is deprecated. pip 25.1 wil

In [2]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm


In [3]:
DATA_DIR = "NLP/FCS"
CSV_DIR = os.path.join(DATA_DIR, "data")

def load_dataset(split='train'):
    df = pd.read_csv(os.path.join(CSV_DIR, f"{split}.csv"))
    df['intent'] = df['action'] + "_" + df['object'] + "_" + df['location']
    df['filepath'] = df['path'].apply(lambda x: os.path.join(DATA_DIR, x))
    return df[['filepath', 'intent']]

train_df = load_dataset('train_data')
val_df = load_dataset('valid_data')
test_df = load_dataset('test_data')

le = LabelEncoder()
y_train = le.fit_transform(train_df['intent'])
y_val = le.transform(val_df['intent'])
y_test = le.transform(test_df['intent'])
num_classes = len(le.classes_)


In [4]:
AUGMENT = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    TimeStretch(min_rate=0.8, max_rate=1.25, p=0.3),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
    Shift(min_shift=-0.2, max_shift=0.2, p=0.3),
])

def extract_melspectrogram(file_path, augment=False, sr=16000, n_mels=64, duration=2.0):
    try:
        y, sr = librosa.load(file_path, sr=sr, duration=duration)
        if augment:
            y = AUGMENT(samples=y, sample_rate=sr)
        target_len = int(sr * duration)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len]
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - np.mean(mel_db)) / (np.std(mel_db) + 1e-6)
        return mel_db.astype(np.float32)
    except Exception as e:
        print(f"Error with file {file_path}: {e}")
        hop_length = 512
        n_frames = int(np.ceil((sr * duration) / hop_length))
        return np.zeros((n_mels, n_frames), dtype=np.float32)


In [5]:
class AudioIntentDataset(Dataset):
    def __init__(self, df, labels, augment=False, sr=16000, n_mels=64, duration=2.0):
        self.df = df.reset_index(drop=True)
        self.labels = labels
        self.augment = augment
        self.sr = sr
        self.n_mels = n_mels
        self.duration = duration

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_path = self.df.loc[idx, 'filepath']
        label = self.labels[idx]
        mel = extract_melspectrogram(file_path, augment=self.augment, sr=self.sr, n_mels=self.n_mels, duration=self.duration)
        # Shape: [n_mels, time]
        return torch.tensor(mel), torch.tensor(label)


In [6]:
def collate_fn(batch):
    # Pads to max time dim in batch
    mels, labels = zip(*batch)
    lengths = [mel.shape[1] for mel in mels]
    max_len = max(lengths)
    padded = []
    for mel in mels:
        if mel.shape[1] < max_len:
            pad_width = max_len - mel.shape[1]
            mel = F.pad(mel, (0, pad_width), 'constant', 0)
        padded.append(mel)
    mels = torch.stack(padded)
    labels = torch.tensor(labels)
    return mels, labels


In [7]:
BATCH_SIZE = 64

train_ds = AudioIntentDataset(train_df, y_train, augment=True)
val_ds = AudioIntentDataset(val_df, y_val, augment=False)
test_ds = AudioIntentDataset(test_df, y_test, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


In [8]:
class CNNAudioGRU(nn.Module):
    def __init__(self, num_classes, input_channels=1):
        super(CNNAudioGRU, self).__init__()
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(0.5)
        self.gru_input_size = 1024
        self.gru = nn.GRU(
            input_size=self.gru_input_size,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.5
        )
        self.attention = nn.Linear(512, 1)
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        # x: [batch, n_mels, time]
        if x.ndim == 3:
            x = x.unsqueeze(1)  # [batch, 1, n_mels, time]
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2).contiguous()  # [batch, time, channels, freq]
        x = x.view(b, w, c*h)  # [batch, time, features]
        x, _ = self.gru(x)
        attn_weights = F.softmax(self.attention(x), dim=1)
        x = torch.sum(x * attn_weights, dim=1)
        x = self.fc(x)
        return x


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNAudioGRU(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(train):
        for mels, labels in tqdm(loader, leave=False):
            mels = mels.to(device)
            labels = labels.to(device)
            outputs = model(mels)
            loss = criterion(outputs, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * mels.size(0)
            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total += mels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / total
    avg_acc = total_correct / total
    return avg_loss, avg_acc, all_preds, all_labels

best_val_acc = 0
for epoch in range(1, 31):
    print(f"Epoch {epoch}")
    train_loss, train_acc, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_acc, _, _ = run_epoch(val_loader, train=False)
    print(f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | Val loss: {val_loss:.4f} acc: {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("✔️ New best model saved.")


Epoch 1


Train loss: 2.2436 acc: 0.2963 | Val loss: 1.5519 acc: 0.4538
✔️ New best model saved.
Epoch 2


Train loss: 1.2504 acc: 0.5626 | Val loss: 0.9687 acc: 0.6652
✔️ New best model saved.
Epoch 3


Train loss: 0.8748 acc: 0.7073 | Val loss: 0.7207 acc: 0.7668
✔️ New best model saved.
Epoch 4


Train loss: 0.6872 acc: 0.7730 | Val loss: 0.6788 acc: 0.7765
✔️ New best model saved.
Epoch 5


...

Train loss: 0.2033 acc: 0.9293 | Val loss: 0.4055 acc: 0.8788
Epoch 27


Train loss: 0.1956 acc: 0.9323 | Val loss: 0.4864 acc: 0.8730
Epoch 28


Train loss: 0.1902 acc: 0.9339 | Val loss: 0.4297 acc: 0.8884
✔️ New best model saved.
Epoch 29


Train loss: 0.1894 acc: 0.9342 | Val loss: 0.4153 acc: 0.8897
✔️ New best model saved.
Epoch 30


Train loss: 0.1765 acc: 0.9389 | Val loss: 0.4751 acc: 0.8727


In [10]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
_, _, test_preds, test_labels = run_epoch(test_loader, train=False)
print("Test accuracy:", accuracy_score(test_labels, test_preds))
print(classification_report(test_labels, test_preds, target_names=le.classes_))


Test accuracy: 0.9348800421829686
                              precision    recall  f1-score   support

          activate_lamp_none       0.95      0.86      0.90        70
     activate_lights_bedroom       0.97      0.91      0.94        97
     activate_lights_kitchen       0.91      0.94      0.93       133
        activate_lights_none       0.91      0.95      0.93        87
    activate_lights_washroom       0.93      0.99      0.96       155
         activate_music_none       0.98      1.00      0.99       121
            bring_juice_none       0.98      1.00      0.99        65
        bring_newspaper_none       0.99      0.99      0.99        79
            bring_shoes_none       0.98      0.99      0.98        89
            bring_socks_none       0.98      0.98      0.98        83
change language_Chinese_none       0.72      0.80      0.76        75
change language_English_none       0.68      0.70      0.69        57
 change language_German_none       0.73      0.69      

In [11]:
import pickle

# Save after training
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
